# ME2021 – Mechanics of Machines I  
## Introduction to Mechanisms and Kinematics

*Instructor:* Thilina H. Weerakkody

## Learning outcomes

By the end of this class, you should be able to:

1. Define **link**, **joint**, **mechanism**, and **degree of freedom (DOF)**.
2. Use **Gruebler’s equation** to compute the mobility of simple planar mechanisms.
3. State **Grashof’s criterion** and apply it to a four–bar linkage.
4. Predict qualitatively how changing link lengths affects whether a link can rotate fully or only oscillate.


## 1. Mechanisms around you

In the lecture slides, you saw examples such as cams operating valves, punch mechanisms, dump truck linkages, toggle clamps, lift tables, and wiper mechanisms. These are all built from **links** connected by **joints** to form a **mechanism**.

**Key reminders:**

- A **link** (or member) is a rigid body.
- A **joint** (or kinematic pair) allows relative motion between links (e.g., revolute, prismatic).
- A **mechanism** is a set of links connected by joints that transforms motion and force.
- A **kinematic chain** is a series of links joined together. If one link is fixed to ground and motion is constrained, it becomes a mechanism.


<p align="center">
  <img src="https://github.com/thilinahwe/thilinahwe.github.io/blob/main/public/Teaching/ME2021/Robot_Joints.png?raw=true" width="400">
</p>


### ✏️ Task 0 – Warm‑up reflection

1. Think of **one mechanism from daily life** (other than those in the slides):  
   - Example: bicycle pedal and crank, office chair height adjustment, car trunk hinge, etc.  
2. In your own words, briefly describe:
   - (a) What are the **links**?
   - (b) What are the **joints**?
   - (c) What is the **input motion** and what is the **output motion**?

👉 Write your answer in the cell below (replace the placeholder text).


_Your answer for Task 0 (you can edit this cell):_

- Mechanism:
- Links:
- Joints:
- Input motion:
- Output motion:


## 2. Mobility of planar mechanisms – Gruebler’s Equation

For a planar mechanism, the **mobility** (number of degrees of freedom) can be estimated using **Gruebler’s equation**:

$$M = 3(n - 1) - 2j_p - j_h$$


where:

- $M$ = mobility of the planar mechanism (number of independent inputs),
- $n$ = total number of links (**including the ground**),
- $j_p$ = number of lower pairs (revolute or prismatic joints),
- $j_h$ = number of higher pairs (e.g., cam or gear contacts).

In many simple mechanisms we have only lower pairs, so often $j_h = 0$.


In [1]:
# @title 🔒 Animation class

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import (
    FloatSlider, IntSlider, Play, jslink,
    VBox, HBox, Output, HTML as WHTML, ToggleButtons
)
from IPython.display import display

# -------------------------------------------
# Grashof classification (4-bar)
# -------------------------------------------
def classify_grashof(L1, L2, L3, L4):
    L = np.array([L1, L2, L3, L4])
    s = np.min(L)
    l = np.max(L)
    mids = sorted(L[(L != s) & (L != l)])
    if len(mids) < 2:
        return "Degenerate / Non-Grashof", "gray"
    p, q = mids

    if s + l < p + q:
        if np.isclose(L1, s):
            return "Grashof Type-1: Crank–rocker", "dodgerblue"
        else:
            return "Grashof Type-1: Double crank", "seagreen"
    else:
        return "Non-Grashof: Double rocker", "crimson"

# -------------------------------------------
# Four-bar geometry
# -------------------------------------------
def fourbar_positions(theta, L1, L2, L3, L4):
    O = np.array([0.0, 0.0])
    A = np.array([L1, 0.0])
    B = np.array([L2*np.cos(theta), L2*np.sin(theta)])

    d = np.linalg.norm(A - B)
    if d == 0 or d > L3 + L4 or d < abs(L3 - L4):
        return O, A, B, None

    a = (L3**2 - L4**2 + d**2) / (2*d)
    h2 = L3**2 - a**2
    if h2 < 0:
        return O, A, B, None

    h = np.sqrt(h2)
    P = B + a*(A - B)/d
    offset = h*np.array([(A[1] - B[1])/d, -(A[0] - B[0])/d])
    C = P + offset
    return O, A, B, C

# -------------------------------------------
# Slider–crank geometry
# -------------------------------------------
def slider_crank_positions(theta, R, L):
    """
    Crank of length R, connecting rod length L, slider on x-axis.
    """
    O = np.array([0.0, 0.0])
    B = np.array([R*np.cos(theta), R*np.sin(theta)])

    By = B[1]
    inside = L**2 - By**2
    if inside < 0:
        return O, B, None

    x = B[0] + np.sqrt(inside)
    C = np.array([x, 0.0])
    return O, B, C

# -------------------------------------------
# Dump-truck geometry (simple sketch model)
# -------------------------------------------
def dump_truck_positions(phi, base_L, bed_L, cyl_base_x, bed_attach_frac):
    """
    - Base: chassis from (0,0) to (base_L, 0)
    - Bed: pivot at (0,0), length = bed_L, angle phi
    - Cylinder: from (cyl_base_x, 0) to attachment on bed
    """
    G1 = np.array([0.0, 0.0])
    G2 = np.array([base_L, 0.0])

    bed_tip = np.array([bed_L*np.cos(phi), bed_L*np.sin(phi)])
    attach = np.array([
        bed_attach_frac*bed_L*np.cos(phi),
        bed_attach_frac*bed_L*np.sin(phi)
    ])
    cyl_base = np.array([cyl_base_x, 0.0])

    return G1, G2, bed_tip, cyl_base, attach

# -------------------------------------------
# Helper: common control legends
# -------------------------------------------
def make_button_legend():
    return WHTML(
        "<div style='font-size:14px; margin:4px 0 6px 2px;'>"
        "<b>Play bar buttons:</b> "
        "<span style='display:inline-block; width:20px; text-align:center;'>▶</span> Play &nbsp;&nbsp; "
        "<span style='display:inline-block; width:20px; text-align:center;'>⏸</span> Pause back &nbsp;&nbsp; "
        "<span style='display:inline-block; width:20px; text-align:center;'>⏮</span> Start &nbsp;&nbsp; "
        "<span style='display:inline-block; width:20px; text-align:center;'>🔁</span> Loop"
        "</div>"
    )

def make_controls_help(extra=""):
    return WHTML(
        "<b>Controls:</b> drag <b>θ</b> slider to scrub manually &nbsp;|&nbsp; "
        "<b>Speed (ms)</b>: smaller = faster, larger = slower"
        + (f" &nbsp;|&nbsp; {extra}" if extra else "")
    )

# -------------------------------------------
# Widget 1: Four-bar with Grashof classification
# -------------------------------------------
def make_fourbar_grashof_widget():
    angle = IntSlider(value=0, min=0, max=360, step=2,
                      description="θ (deg)")
    play = Play(value=0, min=0, max=360, step=2,
                description="", interval=60)
    jslink((play, "value"), (angle, "value"))

    speed = IntSlider(value=60, min=10, max=200, step=10,
                      description="Speed (ms)")

    def update_speed(change):
        play.interval = change["new"]
    speed.observe(update_speed, names="value")

    # Once / Loop / Reflect toggle
    mode = ToggleButtons(
        options=[("Once", "once"), ("Loop", "loop"), ("Reflect", "reflect")],
        value="once",
        description="Mode:",
        style={"button_width": "80px"}
    )

    def update_mode(change):
        m = change["new"]
        if m == "once":
            play.repeat = False
            play._repeat = False
            play._reflect = False
        elif m == "loop":
            play.repeat = True
            play._repeat = True
            play._reflect = False
        elif m == "reflect":
            play.repeat = True
            play._repeat = True
            play._reflect = True
    mode.observe(update_mode, names="value")

    buttons_help = make_button_legend()
    controls_help = make_controls_help("Mode: Once / Loop / Reflect")

    # link length sliders
    L1 = FloatSlider(value=2.0, min=0.5, max=5.0, step=0.1,
                     description="L1 ground (black)")
    L2 = FloatSlider(value=3.0, min=0.5, max=5.0, step=0.1,
                     description="L2 crank (blue)")
    L3 = FloatSlider(value=4.0, min=0.5, max=6.0, step=0.1,
                     description="L3 coupler (green)")
    L4 = FloatSlider(value=3.5, min=0.5, max=6.0, step=0.1,
                     description="L4 output (orange)")

    status = WHTML("")
    info = WHTML(
        "<ul style='margin-top:4px'>"
        "<li><b>Double crank</b> (green text): both blue & orange links rotate fully.</li>"
        "<li><b>Crank–rocker</b> (blue text): blue rotates fully, orange rocks.</li>"
        "<li><b>Double rocker</b> (red text): no full rotations – limited motion.</li>"
        "</ul>"
    )

    out = Output()

    def redraw(*args):
        with out:
            out.clear_output(wait=True)
            gtype, color = classify_grashof(L1.value, L2.value, L3.value, L4.value)
            status.value = f"<h3 style='color:{color}; margin-bottom:2px'>{gtype}</h3>"

            θ = np.radians(angle.value)
            O, A, B, C = fourbar_positions(θ, L1.value, L2.value, L3.value, L4.value)

            fig, ax = plt.subplots(figsize=(5.5, 5.5))
            ax.set_aspect("equal")
            ax.set_xlim(-5, 8)
            ax.set_ylim(-5, 5)
            ax.set_xlabel("x")
            ax.set_ylabel("y")

            # L1 & L2
            ax.plot([O[0], A[0]], [O[1], A[1]], color="black", lw=3, label="L1 ground")
            ax.plot([O[0], B[0]], [O[1], B[1]], color="tab:blue", lw=3, label="L2 crank")

            stuck_msg = None
            if C is None:
                stuck_msg = "Configuration not reachable – mechanism locked for this angle."
            else:
                ax.plot([B[0], C[0]], [B[1], C[1]],
                        color="tab:green", lw=3, label="L3 coupler")
                ax.plot([A[0], C[0]], [A[1], C[1]],
                        color="orange", lw=3, label="L4 output")

                ax.plot(O[0], O[1], "ko", ms=6)
                ax.plot(A[0], A[1], "ko", ms=6)
                ax.plot(B[0], B[1], "ko", ms=6)
                ax.plot(C[0], C[1], "ko", ms=6)

            title = f"Four-bar – θ = {angle.value}°"
            if stuck_msg:
                title += "\n" + stuck_msg
            elif "Double rocker" in gtype:
                title += "\n(Double rocker: only a limited swing is feasible.)"

            ax.set_title(title)
            ax.legend(loc="upper right", fontsize=8)
            plt.show()

    for w in (angle, L1, L2, L3, L4):
        w.observe(redraw, names="value")

    redraw()

    ui = VBox([
        HBox([play, angle, speed]),
        buttons_help,
        mode,
        controls_help,
        L1, L2, L3, L4,
        status,
        info,
        out
    ])
    display(ui)

# -------------------------------------------
# Widget 2: Slider–crank mechanism
# -------------------------------------------
def make_slidercrank_widget():
    angle = IntSlider(value=0, min=0, max=180, step=2,
                      description="θ (deg)")
    play = Play(value=0, min=0, max=180, step=2,
                description="", interval=60)
    jslink((play, "value"), (angle, "value"))

    speed = IntSlider(value=60, min=10, max=200, step=10,
                      description="Speed (ms)")

    def update_speed(change):
        play.interval = change["new"]
    speed.observe(update_speed, names="value")

    buttons_help = make_button_legend()
    controls_help = make_controls_help()

    R = FloatSlider(value=2.0, min=0.5, max=4.0, step=0.1,
                    description="R crank")
    L = FloatSlider(value=5.0, min=2.0, max=8.0, step=0.1,
                    description="L rod")

    out = Output()

    def redraw(*args):
        with out:
            out.clear_output(wait=True)

            θ = np.radians(angle.value)
            O, B, C = slider_crank_positions(θ, R.value, L.value)

            fig, ax = plt.subplots(figsize=(6, 4))
            ax.set_aspect("equal")
            ax.set_xlim(-4, 8)
            ax.set_ylim(-4, 4)
            ax.set_xlabel("x")
            ax.set_ylabel("y")

            ax.plot([-1, 8], [0, 0], "k--", lw=1)

            ax.plot([O[0], B[0]], [O[1], B[1]], "tab:blue", lw=3, label="Crank")
            if C is not None:
                ax.plot([B[0], C[0]], [B[1], C[1]], "tab:green", lw=3, label="Connecting rod")
                ax.plot(C[0], C[1], "ro", ms=6, label="Slider")
            else:
                ax.text(0.5, 0.9, "Configuration not reachable",
                        transform=ax.transAxes, color="red")

            ax.plot(O[0], O[1], "ko", ms=6)
            ax.plot(B[0], B[1], "ko", ms=6)

            ax.set_title(f"Slider–crank – θ = {angle.value}°")
            ax.legend(loc="upper right", fontsize=8)
            plt.show()

    for w in (angle, R, L):
        w.observe(redraw, names="value")

    redraw()

    ui = VBox([
        HBox([play, angle, speed]),
        buttons_help,
        controls_help,
        R, L,
        out
    ])
    display(ui)

# -------------------------------------------
# Widget 3: Dump-truck linkage
# -------------------------------------------
def make_dumptruck_widget():
    angle = IntSlider(value=10, min=0, max=70, step=2,
                      description="Bed angle (deg)")
    play = Play(value=10, min=0, max=70, step=2,
                description="", interval=80)
    jslink((play, "value"), (angle, "value"))

    speed = IntSlider(value=80, min=10, max=200, step=10,
                      description="Speed (ms)")

    def update_speed(change):
        play.interval = change["new"]
    speed.observe(update_speed, names="value")

    buttons_help = make_button_legend()
    controls_help = make_controls_help()

    base_L = FloatSlider(value=6.0, min=4.0, max=10.0, step=0.5,
                         description="Base length")
    bed_L  = FloatSlider(value=5.0, min=3.0, max=8.0, step=0.5,
                         description="Bed length")
    cyl_base_x = FloatSlider(value=3.0, min=1.0, max=8.0, step=0.5,
                             description="Cylinder base x")
    bed_attach_frac = FloatSlider(value=0.6, min=0.3, max=0.9, step=0.05,
                                  description="Bed attach frac")

    out = Output()

    def redraw(*args):
        with out:
            out.clear_output(wait=True)
            φ = np.radians(angle.value)

            G1, G2, bed_tip, cyl_base, attach = dump_truck_positions(
                φ, base_L.value, bed_L.value, cyl_base_x.value, bed_attach_frac.value
            )

            fig, ax = plt.subplots(figsize=(6, 4))
            ax.set_aspect("equal")
            ax.set_xlim(-1, base_L.value + 2)
            ax.set_ylim(-1, bed_L.value + 3)
            ax.set_xlabel("x")
            ax.set_ylabel("y")

            ax.plot([G1[0], G2[0]], [G1[1], G2[1]], "k-", lw=4, label="Chassis")
            ax.plot([G1[0], bed_tip[0]], [G1[1], bed_tip[1]],
                    "s-", lw=4, color="sienna", label="Bed")
            ax.plot([cyl_base[0], attach[0]], [cyl_base[1], attach[1]],
                    "c-", lw=3, label="Hydraulic cylinder")

            ax.plot(G1[0], G1[1], "ko", ms=6)
            ax.plot(G2[0], G2[1], "ko", ms=6)
            ax.plot(cyl_base[0], cyl_base[1], "ko", ms=6)
            ax.plot(attach[0], attach[1], "ko", ms=6)

            ax.set_title(f"Dump truck linkage – bed angle = {angle.value}°")
            ax.legend(loc="upper left", fontsize=8)
            plt.show()

    for w in (angle, base_L, bed_L, cyl_base_x, bed_attach_frac):
        w.observe(redraw, names="value")

    redraw()

    ui = VBox([
        HBox([play, angle, speed]),
        buttons_help,
        controls_help,
        base_L, bed_L, cyl_base_x, bed_attach_frac,
        out
    ])
    display(ui)

# -------------------------------------------
# Quick-return (crank–rocker) geometry
# (reuse the general 4-bar solver)
# -------------------------------------------
def quick_return_positions(theta, a, R, Lc, S):
    """
    Quick-return modelled as a crank–rocker 4-bar:

    a  = ground distance between crank and rocker pivots (O2–O4)
    R  = crank length (O2–A)
    Lc = coupler length (A–B)
    S  = rocker length (O4–B)

    We reuse fourbar_positions(theta, L1, L2, L3, L4) with:
      L1 = a  (ground)
      L2 = R  (crank)
      L3 = Lc (coupler)
      L4 = S  (rocker)
    """
    O2, O4, A, B = fourbar_positions(theta, a, R, Lc, S)
    return O2, O4, A, B   # rename for clarity


# -------------------------------------------
# Quick-return mechanism widget
# -------------------------------------------
def make_quickreturn_widget():
    # Crank angle + play + speed
    angle = IntSlider(value=0, min=0, max=360, step=2,
                      description="θ (deg)")
    play = Play(value=0, min=0, max=360, step=2,
                description="", interval=60)
    jslink((play, "value"), (angle, "value"))

    speed = IntSlider(value=60, min=10, max=200, step=10,
                      description="Speed (ms)")

    def update_speed(change):
        play.interval = change["new"]
    speed.observe(update_speed, names="value")

    # Legends (re-use your helpers)
    buttons_help = make_button_legend()
    controls_help = make_controls_help("Quick-return crank–rocker")

    # Link lengths – defaults chosen to give a crank–rocker Grashof 4-bar
    a_slider  = FloatSlider(value=6.0, min=3.0, max=10.0, step=0.1,
                            description="a ground (black)")
    R_slider  = FloatSlider(value=2.0, min=0.5, max=5.0, step=0.1,
                            description="R crank (blue)")
    Lc_slider = FloatSlider(value=5.0, min=1.0, max=8.0, step=0.1,
                            description="Lc coupler (green)")
    S_slider  = FloatSlider(value=4.0, min=1.0, max=8.0, step=0.1,
                            description="S rocker (orange)")

    info = WHTML(
        "<ul style='margin-top:4px'>"
        "<li>This is a <b>crank–rocker</b> 4-bar used in quick-return mechanisms "
        "(e.g., shapers, slotters).</li>"
        "<li>For one turn of the crank, the rocker swings forward and backward with "
        "different crank-angle ranges, giving a <b>faster return stroke</b>.</li>"
        "</ul>"
    )

    status = WHTML("")
    out = Output()

    def redraw(*args):
        with out:
            out.clear_output(wait=True)

            θ = np.radians(angle.value)
            O2, O4, A, B = quick_return_positions(
                θ,
                a_slider.value,
                R_slider.value,
                Lc_slider.value,
                S_slider.value
            )

            fig, ax = plt.subplots(figsize=(6, 4.8))
            ax.set_aspect("equal")
            ax.set_xlim(-2, a_slider.value + 2)
            ax.set_ylim(-4, 4)
            ax.set_xlabel("x")
            ax.set_ylabel("y")

            locked = False
            if B is None or np.any(np.isnan(B)):
                locked = True

            # Ground link (O2–O4)
            ax.plot([O2[0], O4[0]], [O2[1], O4[1]],
                    color="black", lw=3, label="Ground a")

            if not locked:
                # Crank O2–A
                ax.plot([O2[0], A[0]], [O2[1], A[1]],
                        color="tab:blue", lw=3, label="Crank R")
                # Coupler A–B
                ax.plot([A[0], B[0]], [A[1], B[1]],
                        color="tab:green", lw=3, label="Coupler Lc")
                # Rocker O4–B
                ax.plot([O4[0], B[0]], [O4[1], B[1]],
                        color="orange", lw=3, label="Rocker S")

                # Joints
                ax.plot([O2[0], O4[0], A[0], B[0]],
                        [O2[1], O4[1], A[1], B[1]],
                        "ko", ms=6)
            else:
                ax.text(0.5, 0.5,
                        "Configuration not reachable\n(mechanism locked)",
                        ha="center", va="center",
                        transform=ax.transAxes, color="red", fontsize=11)

            title = f"Quick-return crank–rocker – θ = {angle.value}°"
            ax.set_title(title)
            ax.legend(loc="upper right", fontsize=8)
            plt.show()

            if locked:
                status.value = "<span style='color:red;'>Geometry infeasible for these lengths at this angle.</span>"
            else:
                status.value = ""

    # Re-draw whenever angle or lengths change
    for w in (angle, a_slider, R_slider, Lc_slider, S_slider):
        w.observe(redraw, names="value")

    redraw()

    ui = VBox([
        HBox([play, angle, speed]),
        buttons_help,
        controls_help,
        a_slider, R_slider, Lc_slider, S_slider,
        status,
        info,
        out
    ])
    display(ui)


In [2]:
# @title 🔒 Sim class

# -------------------------------------------
# Slider–crank geometry
# -------------------------------------------
def slider_crank_positions(theta, R, L):
    """
    Simple slider–crank:
    - Crank pivot O at (0, 0)
    - Crank of length R from O to B
    - Connecting rod of length L from B to slider at C on x-axis
    """
    O = np.array([0.0, 0.0])
    B = np.array([R * np.cos(theta), R * np.sin(theta)])

    # Slider is on x-axis at C = (x, 0), with |BC| = L
    By = B[1]
    inside = L**2 - By**2
    if inside < 0:
        # infeasible: rod too short/long for this angle
        return O, B, None

    x = B[0] + np.sqrt(inside)
    C = np.array([x, 0.0])
    return O, B, C


# -------------------------------------------
# Slider–crank widget
# -------------------------------------------
def make_slidercrank_widget():
    angle = IntSlider(value=0, min=0, max=180, step=2,
                      description="θ (deg)")
    play = Play(value=0, min=0, max=180, step=2,
                description="", interval=60)
    jslink((play, "value"), (angle, "value"))

    speed = IntSlider(value=60, min=10, max=200, step=10,
                      description="Speed (ms)")

    def update_speed(change):
        play.interval = change["new"]
    speed.observe(update_speed, names="value")

    # Small legend for the play controls
    buttons_help = WHTML(
        "<div style='font-size:14px; margin:4px 0 6px 2px;'>"
        "<b>Play controls:</b> "
        "left grey squares = ▶ Play / ⏸ Pause / ⏮ Start / 🔁 Loop."
        "</div>"
    )

    controls_help = WHTML(
        "<b>Controls:</b> drag <b>θ (deg)</b> to scrub manually &nbsp;|&nbsp; "
        "<b>Speed (ms)</b>: smaller = faster."
    )

    # Link sliders
    R = FloatSlider(value=2.0, min=0.5, max=4.0, step=0.1,
                    description="R (crank, blue)")
    L = FloatSlider(value=4.0, min=1.0, max=8.0, step=0.1,
                    description="L (rod, green)")

    out = Output()
    status = WHTML("")

    def redraw(*args):
        with out:
            out.clear_output(wait=True)

            theta = np.radians(angle.value)
            O, B, C = slider_crank_positions(theta, R.value, L.value)

            fig, ax = plt.subplots(figsize=(6, 4))
            ax.set_aspect("equal")
            ax.set_xlim(-3, 8)
            ax.set_ylim(-3, 3)
            ax.set_xlabel("x")
            ax.set_ylabel("y")

            # Ground: slider track on x-axis
            ax.plot([-3, 8], [0, 0], "k--", lw=1)

            # Crank
            ax.plot([O[0], B[0]], [O[1], B[1]],
                    color="tab:blue", lw=3, label="Crank R")

            locked = False
            if C is None:
                locked = True
            else:
                # Connecting rod
                ax.plot([B[0], C[0]], [B[1], C[1]],
                        color="tab:green", lw=3, label="Rod L")
                # Slider block
                ax.plot(C[0], C[1], "s", color="orange", ms=10, label="Slider")

                # joints
                ax.plot(O[0], O[1], "ko", ms=6)
                ax.plot(B[0], B[1], "ko", ms=6)

            title = f"Slider–crank – θ = {angle.value}°"
            if locked:
                title += "\nConfiguration not reachable for this R, L, θ."

            ax.set_title(title)
            ax.legend(loc="upper right", fontsize=8)
            plt.show()

            if locked:
                status.value = "<span style='color:red;'>Geometry infeasible: mechanism locked.</span>"
            else:
                status.value = ""

    for w in (angle, R, L):
        w.observe(redraw, names="value")

    redraw()

    ui = VBox([
        HBox([play, angle, speed]),
        buttons_help,
        controls_help,
        R, L,
        status,
        out
    ])
    display(ui)

# -------------------------------------------
# Dump truck geometry (simple sketch model)
# -------------------------------------------
def dump_truck_positions(phi, base_L, bed_L, cyl_base_x, bed_attach_frac):
    """
    Simple dump-truck-like mechanism for visualization:

    - Base (chassis) from G1=(0,0) to G2=(base_L,0)
    - Bed pivots at G1, length bed_L, angle = phi (radians)
    - Cylinder base at (cyl_base_x, 0)
    - Cylinder attaches to bed at bed_attach_frac * bed_L from pivot
    """
    G1 = np.array([0.0, 0.0])
    G2 = np.array([base_L, 0.0])

    bed_tip = np.array([
        bed_L * np.cos(phi),
        bed_L * np.sin(phi)
    ])
    attach = np.array([
        bed_attach_frac * bed_L * np.cos(phi),
        bed_attach_frac * bed_L * np.sin(phi)
    ])

    cyl_base = np.array([cyl_base_x, 0.0])
    return G1, G2, bed_tip, cyl_base, attach


# -------------------------------------------
# Dump truck widget
# -------------------------------------------
def make_dumptruck_widget():
    angle = IntSlider(value=10, min=0, max=70, step=2,
                      description="Bed angle (deg)")
    play = Play(value=10, min=0, max=70, step=2,
                description="", interval=80)
    jslink((play, "value"), (angle, "value"))

    speed = IntSlider(value=80, min=10, max=200, step=10,
                      description="Speed (ms)")

    def update_speed(change):
        play.interval = change["new"]
    speed.observe(update_speed, names="value")

    buttons_help = WHTML(
        "<div style='font-size:14px; margin:4px 0 6px 2px;'>"
        "<b>Play controls:</b> the four grey squares are ▶ Play / ⏸ Pause / ⏮ Start / 🔁 Loop.</div>"
    )

    controls_help = WHTML(
        "<b>Controls:</b> drag <b>Bed angle</b> or press play; adjust <b>Speed (ms)</b>."
    )

    base_L = FloatSlider(value=6.0, min=4.0, max=10.0, step=0.5,
                         description="Base length")
    bed_L  = FloatSlider(value=5.0, min=3.0, max=8.0, step=0.5,
                         description="Bed length")
    cyl_base_x = FloatSlider(value=3.0, min=1.0, max=9.0, step=0.5,
                             description="Cylinder base x")
    bed_attach_frac = FloatSlider(value=0.6, min=0.3, max=0.9, step=0.05,
                                  description="Attach frac")

    out = Output()

    def redraw(*args):
        with out:
            out.clear_output(wait=True)

            phi = np.radians(angle.value)
            G1, G2, bed_tip, cyl_base, attach = dump_truck_positions(
                phi, base_L.value, bed_L.value, cyl_base_x.value, bed_attach_frac.value
            )

            fig, ax = plt.subplots(figsize=(6, 4))
            ax.set_aspect("equal")
            ax.set_xlim(-1, base_L.value + 2)
            ax.set_ylim(-1, bed_L.value + 3)
            ax.set_xlabel("x")
            ax.set_ylabel("y")

            # chassis
            ax.plot([G1[0], G2[0]], [G1[1], G2[1]],
                    "k-", lw=4, label="Chassis")

            # bed
            ax.plot([G1[0], bed_tip[0]], [G1[1], bed_tip[1]],
                    color="sienna", lw=4, label="Bed")

            # cylinder
            ax.plot([cyl_base[0], attach[0]], [cyl_base[1], attach[1]],
                    color="tab:cyan", lw=3, label="Cylinder")

            # joints
            ax.plot(G1[0], G1[1], "ko", ms=6)
            ax.plot(cyl_base[0], cyl_base[1], "ko", ms=6)
            ax.plot(attach[0], attach[1], "ko", ms=6)

            ax.set_title(f"Dump truck linkage – bed angle = {angle.value}°")
            ax.legend(loc="upper left", fontsize=8)
            plt.show()

    for w in (angle, base_L, bed_L, cyl_base_x, bed_attach_frac):
        w.observe(redraw, names="value")

    redraw()

    ui = VBox([
        HBox([play, angle, speed]),
        buttons_help,
        controls_help,
        base_L, bed_L, cyl_base_x, bed_attach_frac,
        out
    ])
    display(ui)


In [3]:
make_fourbar_grashof_widget()

## Key functionalities
 `classify_grashof(L1, L2, L3, L4)`--
Determines the Grashof type of a four-bar:


1. Double crank
2. Crank–rocker
3. Double rocker
4. Degenerate cases



In [4]:
classify_grashof(2, 3, 4, 3.5)

('Grashof Type-1: Crank–rocker', 'dodgerblue')

`fourbar_positions(theta, L1, L2, L3, L4)`


```
O, A, B, C = fourbar_positions(np.radians(30), 2, 3, 4, 3.5)
```





In [5]:
make_slidercrank_widget()

##Dump-truck linkage widget

The dump-truck uses a hydraulic actuator connected to a system of pivoted links that convert the cylinder’s linear extension into a large rotational lift of the truck bed. As the actuator extends, the linkage amplifies motion and raises the bed smoothly while controlling its tipping angle.

<p align="center">
  <img src="https://github.com/thilinahwe/thilinahwe.github.io/blob/main/public/Teaching/ME2021/Dump-Truck.png?raw=true" width="400">
</p>

A hydraulic cylinder pushes on an intermediate link, which amplifies motion and rotates the dump-truck bed about its pivot. The linkage geometry determines how quickly and how far the bed lifts. The simulation illustrates these relationships by showing each link’s contribution to the motion.

In [6]:
make_dumptruck_widget()

### Quick-return mechanism (Whitworth type)

A quick-return mechanism converts uniform crank rotation into a reciprocating motion where the forward (cutting) stroke and the return stroke take different times. By offsetting the slotted lever pivot from the crank center, the mechanism spends a larger crank angle on one stroke and a smaller angle on the return, producing a “quick-return” effect used in shaping and slotting machines.


In [7]:
make_quickreturn_widget()

## 2. Mobility of planar mechanisms – Gruebler’s Equation

For a planar mechanism, the **mobility** (number of degrees of freedom) can be estimated using **Gruebler’s equation**:

$$
M = 3(n - 1) - 2j_p - j_h
$$

where:

- $M$ = mobility of the planar mechanism (number of independent inputs),
- $n$ = total number of links (**including the ground**),
- $j_p$ = number of lower pairs (revolute or prismatic joints),
- $j_h$ = number of higher pairs (e.g., cam or gear contacts).

In many simple mechanisms we have only lower pairs, so often $j_h = 0$.